This is the 5 qubit S-gate injection circuit

In [1]:
import cirq
import numpy as np

In [2]:

# 4 data qubits + 1 ancilla
q0, q1, q2, q3, q4 = cirq.LineQubit.range(5)

circuit = cirq.Circuit()

# --- Prepare ancilla in |T> = T|+> ---
circuit.append([
    cirq.H(q0),
    cirq.S(q0)
])

# --- T gate injection on data qubits ---
circuit.append(cirq.CNOT(q0, q1))
circuit.append(cirq.CNOT(q0, q2))
circuit.append(cirq.CNOT(q0, q3))
circuit.append(cirq.CNOT(q0, q4))

# Measure ancilla
m_anc = cirq.measure(q0, key='m_anc')
circuit.append(m_anc)

# Apply conditional Z correction on data qubit
circuit.append(cirq.Z(q1).with_classical_controls('m_anc'))
circuit.append(cirq.Z(q2).with_classical_controls('m_anc'))
circuit.append(cirq.Z(q3).with_classical_controls('m_anc'))
circuit.append(cirq.Z(q4).with_classical_controls('m_anc'))

# --- (Optional) Other data qubits remain untouched ---
# q1, q2 are spectators in this example

print(circuit)

                                      ┌────┐
0: ───────H───S───@───@───@───@───M────────────
                  │   │   │   │   ║
1: ───────────────X───┼───┼───┼───╫────Z───────
                      │   │   │   ║    ║
2: ───────────────────X───┼───┼───╫────╫Z──────
                          │   │   ║    ║║
3: ───────────────────────X───┼───╫────╫╫Z─────
                              │   ║    ║║║
4: ───────────────────────────X───╫────╫╫╫Z────
                                  ║    ║║║║
m_anc: ═══════════════════════════@════^^^^════
                                      └────┘


Without measurement the unitary is hence:

U_{CS} =
\begin{pmatrix}
T_L & S T_L \\
? & ?
\end{pmatrix}

Note thaat "?" does not matter as we start the ancilla in|0>; based on the measurement we then applay T_L or S T_L

In [12]:
circuit = cirq.Circuit()

# --- Prepare ancilla in |T> = T|+> ---
circuit.append([
    cirq.X(q0),
    cirq.H(q0),
    cirq.S(q0)
])

# --- T gate injection on data qubits ---
circuit.append(cirq.CNOT(q1, q0))
circuit.append(cirq.CNOT(q2, q0))
circuit.append(cirq.CNOT(q3, q0))
circuit.append(cirq.CNOT(q4, q0))

U = cirq.unitary(circuit)

threshold = 1e-10
U[np.abs(U) < threshold] = np.nan
np.set_printoptions(precision=3, 
                    linewidth=500, 
                    threshold=1000000, 
                    edgeitems=10000000, 
                    nanstr=None)
print(np.sqrt(2)*U)

[[ 1. +0.j nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj  1. +0.j nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj]
 [nan+nanj  0. -1.j nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj  0. +1.j nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj]
 [nan+nanj nan+nanj  0. -1.j nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj  0. +1.j nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj]
 [nan+nanj nan+nanj nan+nanj  1. +0.j nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj

In [13]:
#So T_L is
#
# Note: need to renomalise by sqrt 2 for unitary
#
print(np.sqrt(2)*U[:16,:16])

[[ 1. +0.j nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj]
 [nan+nanj  0. -1.j nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj]
 [nan+nanj nan+nanj  0. -1.j nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj]
 [nan+nanj nan+nanj nan+nanj  1. +0.j nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj]
 [nan+nanj nan+nanj nan+nanj nan+nanj  0. -1.j nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj]
 [nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj  1. +0.j nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj]
 [nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj  1. +0.j nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj nan+nanj

In [5]:
S = np.diag([1, 1j])

print(S)

[[1.+0.j 0.+0.j]
 [0.+0.j 0.+1.j]]


In [6]:
#
#   
#
S_alt = np.kron(np.kron(S,S),np.kron(S,S))

threshold = 1e-10
S_alt[np.abs(S_alt) < threshold] = np.nan
print(S_alt)

[[ 1.+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j]
 [nan+0.j  0.+1.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j]
 [nan+0.j nan+0.j  0.+1.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j]
 [nan+0.j nan+0.j nan+0.j -1.+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j]
 [nan+0.j nan+0.j nan+0.j nan+0.j  0.+1.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j]
 [nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j -1.+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j]
 [nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j -1.+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j]
 [nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j nan+0.j -0.-1.j nan+0.j nan+0.j n

In [7]:
#
#   
#
print(np.diagonal(np.sqrt(2)*U[:16,:16]))
print(np.diagonal(S_alt))

#
# S is just S_L = S\otimes S \otimes S\otimes S ?
#
print(np.diagonal(np.sqrt(2)*U[:16,:16]) - np.diagonal(S_alt))

#
#   max diff
#
print(np.max(abs(np.diagonal(np.sqrt(2)*U[:16,:16]) - np.diagonal(S_alt))))
tmp = np.argmax(abs(np.diagonal(np.sqrt(2)*U[:16,:16]) - np.diagonal(S_alt)))
print(tmp, np.sqrt(2)*U[tmp,tmp], np.diagonal(S_alt)[tmp])

[1.+0.j 0.+1.j 0.+1.j 1.+0.j 0.+1.j 1.+0.j 1.+0.j 0.+1.j 0.+1.j 1.+0.j 1.+0.j 0.+1.j 1.+0.j 0.+1.j 0.+1.j 1.+0.j]
[ 1.+0.j  0.+1.j  0.+1.j -1.+0.j  0.+1.j -1.+0.j -1.+0.j -0.-1.j  0.+1.j -1.+0.j -1.+0.j -0.-1.j -1.+0.j -0.-1.j -0.-1.j  1.-0.j]
[2.22e-16+0.00e+00j 0.00e+00+2.22e-16j 0.00e+00+2.22e-16j 2.00e+00+0.00e+00j 0.00e+00+2.22e-16j 2.00e+00+0.00e+00j 2.00e+00+0.00e+00j 0.00e+00+2.00e+00j 0.00e+00+2.22e-16j 2.00e+00+0.00e+00j 2.00e+00+0.00e+00j 0.00e+00+2.00e+00j 2.00e+00+0.00e+00j 0.00e+00+2.00e+00j 0.00e+00+2.00e+00j 2.22e-16+0.00e+00j]
2.0
3 (1.0000000000000002+0j) (-1+0j)


In [8]:
#
#   TO DO
#
#   Can you rewirte the T injection without ancilla but instead with
#   TL = np.sqrt(2)*U[:16,:16])
#
#   For example use: https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.DiagonalGate
#
SL_diago = np.array([1, 1j, 1j, 1,
                    1j, 1, 1, 1j, 
                    1j, 1, 1, 1j,
                    1, 1j, 1j, 1])
    
#    [1, (1+1j)/np.sqrt(2), (1+1j)/np.sqrt(2), 1,
#            (1+1j)/np.sqrt(2), 1, 1, (1+1j)/np.sqrt(2),
#            (1+1j)/np.sqrt(2), 1, 1, (1+1j)/np.sqrt(2),
#            1, (1+1j)/np.sqrt(2), (1+1j)/np.sqrt(2), 1])

print(np.max(abs(np.diagonal(np.sqrt(2)*U[:16,:16]) - SL_diago)))

#tmp = np.argmax(abs(np.diagonal(np.sqrt(2)*U[:16,:16]) - TL_diago))
#print(tmp, np.sqrt(2)*U[tmp,tmp], TL_diago[tmp])

2.220446049250313e-16
